In [1]:
import pandas as pd
df = pd.read_csv('results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [2]:
df.shape

(49547, 9)

In [3]:
df.isnull().sum()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49547 entries, 0 to 49546
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        49547 non-null  str  
 1   home_team   49547 non-null  str  
 2   away_team   49547 non-null  str  
 3   home_score  49547 non-null  int64
 4   away_score  49547 non-null  int64
 5   tournament  49547 non-null  str  
 6   city        49547 non-null  str  
 7   country     49547 non-null  str  
 8   neutral     49547 non-null  bool 
dtypes: bool(1), int64(2), str(6)
memory usage: 3.1 MB


In [7]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'
df['result'] = df.apply(get_result, axis=1)
df['result'].value_counts()

result
home_win    24276
away_win    14010
draw        11261
Name: count, dtype: int64

In [8]:
from sklearn.preprocessing import LabelEncoder

le_home = LabelEncoder()
le_away = LabelEncoder()

df['home_team_enc'] = le_home.fit_transform(df['home_team'])
df['away_team_enc'] = le_away.fit_transform(df['away_team'])
df['neutral_enc'] = df['neutral'].astype(int)

df[['home_team', 'home_team_enc', 'away_team', 'away_team_enc', 'neutral', 'neutral_enc']].head()

,home_team,home_team_enc,away_team,away_team_enc,neutral,neutral_enc
0,Scotland,252,England,89,False,0
1,England,87,Scotland,247,False,0
2,Scotland,252,England,89,False,0
3,England,87,Scotland,247,False,0
4,Scotland,252,England,89,False,0


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Features and target
X = df[['home_team_enc', 'away_team_enc', 'neutral_enc']]
y = df['result']

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit baseline model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.48980827447023206
              precision    recall  f1-score   support

    away_win       0.00      0.00      0.00      2790
        draw       0.00      0.00      0.00      2266
    home_win       0.49      1.00      0.66      4854

    accuracy                           0.49      9910
   macro avg       0.16      0.33      0.22      9910
weighted avg       0.24      0.49      0.32      9910



C:\Users\ASH\DataScience\football-model\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASH\DataScience\football-model\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASH\DataScience\football-model\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: turn team names into yes/no columns instead of numbers
X = pd.get_dummies(df[['home_team', 'away_team', 'neutral_enc']], columns=['home_team', 'away_team'])
y = df['result']

# Step 2: split into training data and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 4: check how well it did
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.5779011099899092
              precision    recall  f1-score   support

    away_win       0.53      0.55      0.54      2790
        draw       0.31      0.04      0.07      2266
    home_win       0.61      0.85      0.71      4854

    accuracy                           0.58      9910
   macro avg       0.48      0.48      0.44      9910
weighted avg       0.52      0.58      0.51      9910

